### Formulation

The propagator is
$$
\begin{gather*}
    K(q_f, q_i, t)=\langle q_f|e^{-i\hat{H}t/\hbar}|q_i\rangle=\int\mathcal{D}q~e^{iS[q]/\hbar}\\
    S[q]=\int dt~\bigg(\frac{1}{2}m\dot{q}^2-V(q)\bigg)
\end{gather*}
$$
Using the Wick rotation, $t=-i\tau$,
$$
\begin{gather*}
    \dot{q}^2=\bigg(\frac{dq}{dt}\bigg)^2=\bigg(\frac{dq}{d\tau}\frac{1}{-i}\bigg)^2=-\bigg(\frac{dq}{d\tau}\bigg)^2\\
    \frac{i}{\hbar}S[q]=\frac{i}{\hbar}\int -id\tau~\bigg(-\frac{1}{2}m\bigg(\frac{dq}{d\tau}\bigg)^2-V(q)\bigg)=\frac{-1}{\hbar}\int d\tau~\bigg(\frac{1}{2}m\bigg(\frac{dq}{d\tau}\bigg)^2+V(q)\bigg)=\frac{-1}{\hbar}S_E[q]\\
    \therefore K_E(q_f, q_i, \tau)=\langle q_f|e^{-\hat{H}\tau/\hbar}|q_i\rangle=\int\mathcal{D}q~e^{-S_E[q]/\hbar}
\end{gather*}
$$
with the Euclidean action $S_E$,
$$
\begin{gather*}
    S_E[q]=\int_0^\tau d\tau~\bigg(\frac{1}{2}m\bigg(\frac{dq}{d\tau}\bigg)^2+V(q(\tau))\bigg)
\end{gather*}
$$
This Formulation implies connection to statistical physics. The partition function is
$$
\begin{gather*}
    Z=\mathrm{tr}(e^{-\beta\hat{H}})=\int dq~\langle q|e^{-\beta\hat{H}}|q\rangle
\end{gather*}
$$
If we replace $\tau=\hbar\beta$,
$$
\begin{gather*}
    Z=\int dq~\langle q|e^{-\beta\hat{H}}|q\rangle=\int dq~K_E(q, q, \hbar\beta)=\int\mathcal{D}q~e^{-S_E[q]/\hbar},\quad q(0)=q(\hbar\beta)
\end{gather*}
$$


In [ ]:
import numpy as np
import numpy.random as nr
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
def V_sho(x, w=0.1):
    # Simple Harmonic potential
    return 0.5(w**2)*(x**2)


def V_higgs(x, a=3, l=10):
    # Higgs potential
    return l*(x**2 - a**2)**2

In [ ]:
class pathIntegral_particle:
    def __init__(self, xi, xf, dt, V):
        self.xi = xi
        self.xf = xf
        self.dt = dt
        self.V = V

        self.x = np.arange(xi, xf+dt, dt)
        self.dx = np.zeros_like(self.x)
        self.N = len(self.x)


    def S_E(self):
        # Euclidean action

        v = np.roll(self.x, -1) - self.x
        S = np.sum(0.5*v*v + self.V(self.x))

        return S
    

    def Metropolis(self, T):
        # Metropolis altorithm

        r = nr.randint(self.N)
        s = nr.rand()

        self.dx[r] += s

        dE = self.S_E(self.x + self.dx) - self.S_E(self.x)

        if dE <= 0:
            # Energy decreasing
            self.x += self.dx

        elif nr.rand() < np.exp(dE/T):
            self.x += self.dx


    def MonteCarlo(self, eq_steps=10**6, MC_steps=10**6):
        #i = 0

        for t in tqdm(self.T):

            for _ in range(eq_steps):
                # Equilibriate steps
                self.Metropolis(t)

            for _ in range(MC_steps):
                # Monte Carlo steps
                self.Metropolis(t)